In [0]:
%sql
-- Populate with unique products from Silver layer
INSERT OVERWRITE TABLE gold_db.dim_products
SELECT 
    -- Generate a unique product ID from sku + brand
    md5(concat(sku, brand)) AS product_id,
    sku AS sku_code,
    name AS product_name,
    brand,
    category_path, 
    categories,
    url as product_url,
    MIN(scraped_at) AS first_seen_date,
    MAX(scraped_at) AS last_seen_date
FROM gold_db.silver_products
WHERE sku IS NOT NULL
GROUP BY sku, name, brand, category_path, categories, url;

In [0]:
#%sql
#select *
#from gold_db.dim_products

In [0]:
%sql
-- Populate price history from Silver layer
MERGE INTO gold_db.fct_prices AS target
USING (
    SELECT 
        md5(concat(sku, CAST(scraped_at AS STRING))) AS price_id,
        md5(concat(sku, brand)) AS product_id,
        DATE(scraped_at) AS scraped_date,
        scraped_at,
        price
    FROM gold_db.silver_products
    WHERE sku IS NOT NULL
) AS source
ON target.price_id = source.price_id
WHEN NOT MATCHED THEN INSERT *;

In [0]:
#%sql
#select *
#from gold_db.fct_prices